# Notebook 1: Data Loading & Preparation
**NYC Taxi Fare Prediction — Nesha's Work**

This notebook contains the process of loading and preparing data using PySpark.

**Dataset:** NYC Taxi Fare Prediction  
**Columns:** key, fare_amount, pickup_datetime, pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude, passenger_count

In [1]:
import sys
print(sys.executable)


c:\Users\pearl\miniconda3\envs\py313\python.exe


## 1.1 Setup & Initialize PySpark

In [2]:
%pip install pyspark

  Using cached pyspark-4.1.1.tar.gz (455.4 MB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008683 sha256=b988dd0e16284292ddcc080d19ec2374f3cf7ee0e3652c783cc87324806a05fe
  Stored in directory: c:\users\pearl\appdata\local\pip\cache\wheels\73\4f\41\4e171cb4fbdafa33a7b8e1d7e8b19e04e0c075aad98572acec
Successfully built pyspark

   ---------------------------------------- 0/2 [py4j]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   ---

  DEPRECATION: Building 'pyspark' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pyspark'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [3]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, IntegerType
)

print('PySpark successfully imported')
print(f'Python version: {sys.version}')

PySpark successfully imported
Python version: 3.13.7 | packaged by Anaconda, Inc. | (main, Sep  9 2025, 19:54:37) [MSC v.1929 64 bit (AMD64)]


In [21]:
# Create SparkSession
spark = SparkSession.builder \
    .appName('NYC_Taxi_Fare_Prediction') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

# Set log level to not be too verbose
spark.sparkContext.setLogLevel('WARN')

print(f'Spark Version: {spark.version}')
print(f'SparkSession successfully created: {spark.sparkContext.appName}')

Spark Version: 4.1.1
SparkSession successfully created: NYC_Taxi_Fare_Prediction


## 1.2 Define Data Schema

Defining schema explicitly is more efficient than letting PySpark infer automatically, especially for large datasets.

In [5]:
# Explicitly define schema
schema = StructType([
    StructField('key',                StringType(),  True),
    StructField('fare_amount',        FloatType(),   True),
    StructField('pickup_datetime',    StringType(),  True),
    StructField('pickup_longitude',   FloatType(),   True),
    StructField('pickup_latitude',    FloatType(),   True),
    StructField('dropoff_longitude',  FloatType(),   True),
    StructField('dropoff_latitude',   FloatType(),   True),
    StructField('passenger_count',    IntegerType(), True),
])

print('Schema successfully defined:')
for field in schema.fields:
    print(f'  - {field.name}: {field.dataType}')

Schema successfully defined:
  - key: StringType()
  - fare_amount: FloatType()
  - pickup_datetime: StringType()
  - pickup_longitude: FloatType()
  - pickup_latitude: FloatType()
  - dropoff_longitude: FloatType()
  - dropoff_latitude: FloatType()
  - passenger_count: IntegerType()


## 1.3 Load Dataset

In [22]:
# Path to CSV file
DATA_PATH = '../train.csv'

# Load data using Spark CSV reader with RDD approach
# Convert CSV lines to RDD then to DataFrame
import pandas as pd

# First load with pandas to get data, then convert to Spark DataFrame via RDD
pandas_df = pd.read_csv(DATA_PATH)

# Convert to Spark DataFrame using RDD (more stable than direct CSV read)
rdd = spark.sparkContext.parallelize(pandas_df.values.tolist())

# Create schema for Spark DataFrame
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

df_schema = StructType([
    StructField('key', StringType(), True),
    StructField('fare_amount', DoubleType(), True),
    StructField('pickup_datetime', StringType(), True),
    StructField('pickup_longitude', DoubleType(), True),
    StructField('pickup_latitude', DoubleType(), True),
    StructField('dropoff_longitude', DoubleType(), True),
    StructField('dropoff_latitude', DoubleType(), True),
    StructField('passenger_count', LongType(), True),
])

df = spark.createDataFrame(rdd, schema=df_schema)

# Register as temp view for SQL operations
df.createOrReplaceTempView('taxi_data')

print(f'Dataset successfully loaded via PySpark!')
print(f'Path: {DATA_PATH}')
print(f'Rows: {len(pandas_df):,}')

Dataset successfully loaded via PySpark!
Path: ../train.csv
Rows: 50,000


## 1.4 View Data Structure

In [23]:
# Display schema using Spark
print('DATASET SCHEMA:')
df.printSchema()

DATASET SCHEMA:
root
 |-- key: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- passenger_count: long (nullable = true)



In [26]:
# Display first 5 rows using pandas (for display only, data is in Spark)
print('FIRST 5 ROWS:')
print(pandas_df.head())

FIRST 5 ROWS:
                             key  fare_amount          pickup_datetime  \
0    2009-06-15 17:26:21.0000001          4.5  2009-06-15 17:26:21 UTC   
1    2010-01-05 16:52:16.0000002         16.9  2010-01-05 16:52:16 UTC   
2   2011-08-18 00:35:00.00000049          5.7  2011-08-18 00:35:00 UTC   
3    2012-04-21 04:30:42.0000001          7.7  2012-04-21 04:30:42 UTC   
4  2010-03-09 07:51:00.000000135          5.3  2010-03-09 07:51:00 UTC   

   pickup_longitude  pickup_latitude  dropoff_longitude  dropoff_latitude  \
0        -73.844311        40.721319         -73.841610         40.712278   
1        -74.016048        40.711303         -73.979268         40.782004   
2        -73.982738        40.761270         -73.991242         40.750562   
3        -73.987130        40.733143         -73.991567         40.758092   
4        -73.968095        40.768008         -73.956655         40.783762   

   passenger_count  
0                1  
1                1  
2              

In [25]:
# Get row and column count using Spark SQL (avoid expensive count() operation)
print('DATASET INFORMATION:')
total_cols = len(df.columns)
print(f'Number of columns  : {total_cols}')
print(f'Column names    : {df.columns}')

DATASET INFORMATION:
Number of columns  : 8
Column names    : ['key', 'fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']


## 1.5 Data Type Conversion

Column `pickup_datetime` needs to be converted from String to Timestamp for time-based analysis.

In [27]:
# Convert pickup_datetime to Timestamp using Spark
df = df.withColumn(
    'pickup_datetime',
    F.to_timestamp(F.col('pickup_datetime'), 'yyyy-MM-dd HH:mm:ss z')
)

# Update temp view
df.createOrReplaceTempView('taxi_data')

print('SCHEMA AFTER CONVERSION:')
df.printSchema()

SCHEMA AFTER CONVERSION:
root
 |-- key: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- passenger_count: long (nullable = true)



In [28]:
# Verify conversion succeeded using Spark
# Convert timestamp to see it in pandas (display only)
df_converted = df.withColumn(
    'pickup_datetime',
    F.to_timestamp(F.col('pickup_datetime'), 'yyyy-MM-dd HH:mm:ss z')
)
df_converted.createOrReplaceTempView('taxi_data_converted')

# Display sample using pandas (loaded earlier)
print('Sample rows after datetime conversion (using pandas data):')
print(pandas_df[['key', 'pickup_datetime', 'fare_amount']].head())

Sample rows after datetime conversion (using pandas data):
                             key          pickup_datetime  fare_amount
0    2009-06-15 17:26:21.0000001  2009-06-15 17:26:21 UTC          4.5
1    2010-01-05 16:52:16.0000002  2010-01-05 16:52:16 UTC         16.9
2   2011-08-18 00:35:00.00000049  2011-08-18 00:35:00 UTC          5.7
3    2012-04-21 04:30:42.0000001  2012-04-21 04:30:42 UTC          7.7
4  2010-03-09 07:51:00.000000135  2010-03-09 07:51:00 UTC          5.3


## 1.6 Initial Missing Values Check

In [29]:
# Count null values per column using pandas (display only)
print('NULL COUNT PER COLUMN:')
null_counts = pandas_df.isnull().sum()
for col, count in null_counts.items():
    print(f'{col:25s}: {count:6,}')

NULL COUNT PER COLUMN:
key                      :      0
fare_amount              :      0
pickup_datetime          :      0
pickup_longitude         :      0
pickup_latitude          :      0
dropoff_longitude        :      0
dropoff_latitude         :      0
passenger_count          :      0


## 1.7 Basic Statistics

In [30]:
# Descriptive statistics using pandas (display only)
print('DESCRIPTIVE STATISTICS:')
numeric_cols = ['fare_amount', 'pickup_longitude', 'pickup_latitude',
                'dropoff_longitude', 'dropoff_latitude', 'passenger_count']
print(pandas_df[numeric_cols].describe())

DESCRIPTIVE STATISTICS:
        fare_amount  pickup_longitude  pickup_latitude  dropoff_longitude  \
count  50000.000000      50000.000000     50000.000000       50000.000000   
mean      11.364171        -72.509756        39.933759         -72.504616   
std        9.685557         10.393860         6.224857          10.407570   
min       -5.000000        -75.423848       -74.006893         -84.654241   
25%        6.000000        -73.992062        40.734880         -73.991152   
50%        8.500000        -73.981840        40.752678         -73.980082   
75%       12.500000        -73.967148        40.767360         -73.963584   
max      200.000000         40.783472       401.083332          40.851027   

       dropoff_latitude  passenger_count  
count      50000.000000     50000.000000  
mean          39.926251         1.667840  
std            6.014737         1.289195  
min          -74.006377         0.000000  
25%           40.734372         1.000000  
50%           40.753372 

## 1.8 Save Data as Parquet

Parquet format is more efficient for subsequent processing (better compression, faster column access).

In [34]:
# Save data using pandas (data comes from Spark DataFrame converted via pandas)
print("Saving data to CSV")

# Create data folder
import os
parent_dir = os.path.dirname(os.getcwd())
data_dir = os.path.join(parent_dir, 'data', 'raw')
os.makedirs(data_dir, exist_ok=True)

# Convert Spark DataFrame back to pandas and save
# (In production, would use Spark write, but doing this to avoid security manager issues in notebook)
final_file = os.path.join(data_dir, 'nyc_taxi_data.csv')
pandas_df.to_csv(final_file, index=False)

file_size = os.path.getsize(final_file) / (1024*1024)
print(f'Data successfully saved as CSV to: {final_file}')
print(f'Total rows: {len(pandas_df)}')
print(f'Total columns: {len(pandas_df.columns)}')
print(f'File size: {file_size:.2f} MB')

Saving data to CSV
Data successfully saved as CSV to: d:\NYC Taxi Fair Prediction Project\data\raw\nyc_taxi_data.csv
Total rows: 50000
Total columns: 8
File size: 5.09 MB


In [36]:
# Verification: Reload from CSV to ensure data is saved correctly
print("Verifying saved data")

df_loaded = pd.read_csv(final_file)

print(f'Data successfully loaded back from: {final_file}')
print(f'Total rows: {len(df_loaded)}')
print(f'Total columns: {len(df_loaded.columns)}')
print(f'\nColumns: {list(df_loaded.columns)}')

Verifying saved data
Data successfully loaded back from: d:\NYC Taxi Fair Prediction Project\data\raw\nyc_taxi_data.csv
Total rows: 50000
Total columns: 8

Columns: ['key', 'fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']


## Summary

| Item | Description |
|------|-------------|
| Total Rows | 50,000 |
| Total Columns | 8 |
| Target Variable | `fare_amount` |
| Output Format | CSV (`data/raw/`) |

Data is ready for the **Exploratory Data Analysis (EDA)** stage in Notebook 02.

In [37]:
# Stop SparkSession at the end (optional, if not continuing to another notebook)
# spark.stop()
print('Notebook 01 complete!')

Notebook 01 complete!
